In [15]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from litellm import completion
from pydantic import BaseModel

from dotenv import load_dotenv
import os
load_dotenv()

True

In [16]:

import json
import time
from typing import Any

class LLMClient:
    def __init__(self, model_name: str, max_tokens: int = 4000):
        self.model_name = model_name
        self.request_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_tokens_used = 0
        self.total_latency_ms = 0.0
        self.last_usage: dict[str, Any] = {}
        self.last_error: str | None = None
        self.last_response = None
        self.max_tokens = max_tokens

    def get_usage(self) -> str:
        return (
            f"Using model: {self.model_name}, Requests: {self.request_count}, "
            f"Prompt tokens: {self.total_prompt_tokens}, Completion tokens: {self.total_completion_tokens}, "
            f"Total tokens used: {self.total_tokens_used}, Total latency ms: {round(self.total_latency_ms, 2)}"
        )

    def get_usage_details(self) -> dict[str, Any]:
        return {
            "model_name": self.model_name,
            "request_count": self.request_count,
            "prompt_tokens": self.total_prompt_tokens,
            "completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_tokens_used,
            "total_latency_ms": round(self.total_latency_ms, 2),
            "last_usage": self.last_usage,
            "last_error": self.last_error,
        }

    def _record_usage(self, response: Any, latency_ms: float) -> None:
        self.request_count += 1
        self.total_latency_ms += latency_ms
        usage = getattr(response, "usage", None)
        if usage:
            prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
            completion_tokens = getattr(usage, "completion_tokens", 0) or 0
            total_tokens = getattr(usage, "total_tokens", prompt_tokens + completion_tokens) or 0
            self.total_prompt_tokens += prompt_tokens
            self.total_completion_tokens += completion_tokens
            self.total_tokens_used += total_tokens
            self.last_usage = {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens,
            }
        else:
            self.last_usage = {}

    def _extract_choice(self, response: Any) -> Any:
        choices = getattr(response, "choices", None)
        if choices and len(choices) > 0:
            return choices[0]
        return None

    def _extract_message(self, response: Any) -> Any:
        choice = self._extract_choice(response)
        return getattr(choice, "message", None) if choice else None

    def _extract_tool_calls(self, message: Any) -> list[dict[str, Any]]:
        tool_calls = getattr(message, "tool_calls", None) or []
        normalized_calls: list[dict[str, Any]] = []
        for tool_call in tool_calls:
            function_call = getattr(tool_call, "function", None)
            normalized_calls.append({
                "id": getattr(tool_call, "id", None),
                "type": getattr(tool_call, "type", None),
                "name": getattr(function_call, "name", None),
                "arguments": getattr(function_call, "arguments", None),
            })
        return normalized_calls

    def _parse_structured_content(self, content: str | None, response_format: type[BaseModel] | None) -> Any:
        if not content:
            return {}

        if response_format and isinstance(response_format, type) and issubclass(response_format, BaseModel):
            try:
                return response_format.model_validate_json(content)
            except Exception:
                pass

        try:
            return json.loads(content)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Structured response was not valid JSON: {exc}") from exc

    def generate(self, messages: list[dict[str, str]], max_tokens: int | None = None, tools: list[dict[str, Any]] | None = None, tool_choice: Any = None, return_full_response: bool = False, temperature: float | None = 0) -> str | dict[str, Any]:
        resolved_max_tokens = self.max_tokens if max_tokens is None else max_tokens
        request_kwargs: dict[str, Any] = {
            "model": self.model_name,
            "messages": messages,
            "max_tokens": resolved_max_tokens,
            "stream": False,
            "temperature": temperature,
        }
        if tools is not None:
            request_kwargs["tools"] = tools
        if tool_choice is not None:
            request_kwargs["tool_choice"] = tool_choice

        start_time = time.perf_counter()
        self.last_error = None
        try:
            response = completion(**request_kwargs)
        except Exception as exc:
            self.last_error = str(exc)
            raise
        latency_ms = (time.perf_counter() - start_time) * 1000
        self.last_response = response
        self._record_usage(response, latency_ms)

        message = self._extract_message(response)
        content = getattr(message, "content", None) if message else None
        tool_calls = self._extract_tool_calls(message) if message else []
        text = content or ("No tool call content returned." if tool_calls else "No content in response.")

        if return_full_response:
            return {
                "content": content,
                "text": text,
                "tool_calls": tool_calls,
                "usage": self.last_usage,
                "latency_ms": round(latency_ms, 2),
                "model_name": self.model_name,
                "raw_response": response,
            }

        return text

    def generate_structured(self, messages: list[dict[str, str]], response_format: type[BaseModel] | None = None, max_tokens: int| None = None, tools: list[dict[str, Any]] | None = None, tool_choice: Any = None, temperature: float | None = 0) -> Any:
        resolved_max_tokens = self.max_tokens if max_tokens is None else max_tokens
        request_kwargs: dict[str, Any] = {
            "model": self.model_name,
            "messages": messages,
            "max_tokens": resolved_max_tokens,
            "stream": False,
            "temperature": temperature,
        }
        if response_format is not None:
            request_kwargs["response_format"] = response_format
        if tools is not None:
            request_kwargs["tools"] = tools
        if tool_choice is not None:
            request_kwargs["tool_choice"] = tool_choice

        start_time = time.perf_counter()
        self.last_error = None
        try:
            response = completion(**request_kwargs)
        except Exception as exc:
            self.last_error = str(exc)
            raise
        latency_ms = (time.perf_counter() - start_time) * 1000
        self.last_response = response
        self._record_usage(response, latency_ms)

        message = self._extract_message(response)
        content = getattr(message, "content", None) if message else None
        if not content:
            return {}
        return self._parse_structured_content(content, response_format)
    

In [17]:

openai_client = LLMClient(model_name="openai/gpt-4o-mini", max_tokens=4000)
groq_client = LLMClient(model_name="groq/openai/gpt-oss-120b", max_tokens=4000)
github_client = LLMClient(model_name="github/Llama-3.3-70B-Instruct", max_tokens=4000)

In [18]:
llm_client = github_client

In [49]:
class TempResponse(BaseModel):
    answer: str
    confidence: float

res=llm_client.generate_structured([{"role": "user", "content": "What is the capital of France?"}], response_format=TempResponse, max_tokens=100)

In [50]:
res.model_dump()

{'answer': 'Paris', 'confidence': 1.0}

In [5]:
llm_client.get_usage()

'Using model: github/Llama-3.3-70B-Instruct, Requests: 1, Prompt tokens: 42, Completion tokens: 8, Total tokens used: 50, Total latency ms: 2190.78'

In [19]:
uri = "lancedb"
db = lancedb.connect(uri)

In [20]:
CONVERSATIONAL_TABLE = "CONVERSATIONAL_MEMORY"
KNOWLEDGE_BASE_TABLE = "SEMANTIC_MEMORY"
WORKFLOW_TABLE = "WORKFLOW_MEMORY"
TOOLBOX_TABLE = "TOOLBOX_MEMORY"
ENTITY_TABLE = "ENTITY_MEMORY"
SUMMARY_TABLE = "SUMMARY_MEMORY"
TOOL_LOG_TABLE = "TOOL_LOG_MEMORY"

In [21]:
func = get_registry().get("huggingface").create(name="colbert-ir/colbertv2.0")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
func.ndims()

768

In [66]:
ALL_TABLES = [
    CONVERSATIONAL_TABLE,
    KNOWLEDGE_BASE_TABLE,
    WORKFLOW_TABLE,
    TOOLBOX_TABLE,
    ENTITY_TABLE,
    SUMMARY_TABLE,
    TOOL_LOG_TABLE]

# clear all tables
for table in ALL_TABLES:
    try:
        db.drop_table(table)
    except Exception as e:
        print(f"Error dropping table {table}: {e}")

In [67]:
from typing import Optional
from datetime import datetime, timezone

from pydantic import Field


class ConversationLog(LanceModel):
    id: Optional[str] = None
    thread_id: str
    role: str
    content: str
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    metadata: Optional[str] = None
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    summary_id: Optional[str] = None


class ToolLog(LanceModel):
    id: Optional[str] = None
    thread_id: str
    tool_call_id: Optional[str] = None
    tool_name: str
    tool_args: Optional[str] = None
    result: Optional[str] = None
    result_preview: Optional[str] = None
    status: str = "success"
    error_message: Optional[str] = None
    metadata: Optional[str] = None
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class MetadataEntry(LanceModel):
    key: str
    value: str

class KnowledgeBaseEntry(LanceModel):
    id: Optional[str] = None
    thread_id: Optional[str] = None
    text: str = func.SourceField()
    vector: Optional[Vector(func.ndims())] = func.VectorField()  # pyright: ignore[reportInvalidTypeForm]
    metadata: Optional[list[MetadataEntry]] = None
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

In [68]:
class StorageManger:
    """
    Responsible for creating tables and setting up getters for each table.
    """
    def __init__(self, db_uri: str):
        self.db = lancedb.connect(db_uri)
        self.conversation_table = db.create_table(CONVERSATIONAL_TABLE, schema=ConversationLog, mode="overwrite")
        self.tool_log_table = db.create_table(TOOL_LOG_TABLE, schema=ToolLog, mode="overwrite")
        self.knowledge_base_table = db.create_table(KNOWLEDGE_BASE_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.workflow_table = db.create_table(WORKFLOW_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.toolbox_table = db.create_table(TOOLBOX_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")
        self.entity_table = db.create_table(ENTITY_TABLE, schema=KnowledgeBaseEntry,  mode="overwrite")
        self.summary_table = db.create_table(SUMMARY_TABLE, schema=KnowledgeBaseEntry, mode="overwrite")

    def get_conversation_table(self):
        return self.db.open_table(CONVERSATIONAL_TABLE)
    
    def get_tool_log_table(self):
        return self.db.open_table(TOOL_LOG_TABLE)
    
    def get_knowledge_base_table(self):
        return self.db.open_table(KNOWLEDGE_BASE_TABLE)
    
    def get_workflow_table(self):
        return self.db.open_table(WORKFLOW_TABLE)
    
    def get_toolbox_table(self):
        return self.db.open_table(TOOLBOX_TABLE)
    
    def get_entity_table(self):
        return self.db.open_table(ENTITY_TABLE)
    
    def get_summary_table(self):
        return self.db.open_table(SUMMARY_TABLE)
    

class MemoryManager:
    """
    Responsible for managing memory operations such as adding entries, retrieving entries, and performing similarity searches.
    """
    def __init__(self, storage_manager: StorageManger):
        self.storage_manager = storage_manager
        self.conversation_table = storage_manager.get_conversation_table()
        self.tool_log_table = storage_manager.get_tool_log_table()
        self.knowledge_base_table = storage_manager.get_knowledge_base_table()
        self.workflow_table = storage_manager.get_workflow_table()
        self.toolbox_table = storage_manager.get_toolbox_table()
        self.entity_table = storage_manager.get_entity_table()
        self.summary_table = storage_manager.get_summary_table()

    def add_conversation_log(self, log: list[ConversationLog]):
        self.conversation_table.add(log)

    def add_tool_log(self, log: list[ToolLog]):
        self.tool_log_table.add(log)
    
    def add_knowledge_base_entry(self, entry: list[KnowledgeBaseEntry]):
        self.knowledge_base_table.add(entry)
    
    def add_workflow_entry(self, entry: list[KnowledgeBaseEntry]):
        self.workflow_table.add(entry)
    
    def add_toolbox_entry(self, entry:  list[KnowledgeBaseEntry]):
        self.toolbox_table.add(entry)

    def read_toolbox_entry(self, query, limit=5):
        results = self.toolbox_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).to_pandas().sort_values("_distance").head(limit)

        tools=[] # should contain all the tool names

        for _, row in results.iterrows():
            metadata_list = row.get("metadata") or []
            metadata_dict = {}
            for entry in metadata_list:
                if isinstance(entry, dict):
                    key = entry.get("key")
                    value = entry.get("value")
                else:
                    key = getattr(entry, "key", None)
                    value = getattr(entry, "value", None)
                if key is not None:
                    metadata_dict[key] = value

            tool_name = metadata_dict.get("tool_name")
            if tool_name:
                tools.append(tool_name)

        return tools
            

        # get tool functions 

    def add_entity_entry(self, entry:  list[KnowledgeBaseEntry]):
        self.entity_table.add(entry)

    def read_knowledge_base_entry(self, query, limit=5):
        self.knowledge_base_table.search(
            query=query,
            query_type="vector",
            vector_column_name="vector",
        ).to_pandas().sort_values("_distance").head(limit)
    
    

storage_manager = StorageManger(uri)
memory_manager = MemoryManager(storage_manager)



[2026-04-12T21:18:57Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/CONVERSATIONAL_MEMORY.lance, it will be created
[2026-04-12T21:18:57Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/TOOL_LOG_MEMORY.lance, it will be created
[2026-04-12T21:18:57Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/SEMANTIC_MEMORY.lance, it will be created
[2026-04-12T21:18:57Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/WORKFLOW_MEMORY.lance, it will be created
[2026-04-12T21:18:57Z WARN  lance::dataset::write::insert] No existing dataset at /Users/gautamnaik/Developer/ai/AdvancedMemoryAgents/notebooks/lancedb/TOOLBOX_MEMORY.lance, it will be created
[2026-04-12T21:18:57Z WAR

In [69]:
import inspect
import json
from typing import Type, Callable, Any, Dict
from pydantic import BaseModel
from openai.lib import pydantic_function_tool

class ToolRegistry:
    def __init__(self, client: LLMClient, memory_manager: MemoryManager):
        self.client = client
        self.memory_manager = memory_manager
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, param_schema: Type[BaseModel]):
        def decorator(func: Callable):
            func_name = func.__name__
            
            # 1. Generate the standard OpenAI Tool Schema
            openai_tool_schema = pydantic_function_tool(param_schema)
            
            # 2. Extract source for augmentation
            source_code = inspect.getsource(func)
            
            # Store initial metadata
            self.tools[func_name] = {
                "callable": func,
                "schema": openai_tool_schema,
                "param_model": param_schema,
                "source_code": source_code,
                "augmented_metadata": None  # Populated in next step
            }


            return func
        return decorator
    
    def _save_tool_to_db(self, tool_name: str):
        tool_data = self.tools[tool_name]
        # Persist metadata values as strings to match LanceDB Arrow-compatible schema.
        # serialized_metadata = [
        #     MetadataEntry(
        #         key=k,
        #         value=json.dumps(v, default=str) if not isinstance(v, str) else v,
        #     )
        #     for k, v in tool_data.items()
        # ]

        entry = KnowledgeBaseEntry(
            text=json.dumps({
                "name": tool_name,
                "schema": tool_data["schema"],
                "augmented_metadata": tool_data["augmented_metadata"],
            }, default=str),
            metadata=[MetadataEntry(key="tool_name", value=tool_name)],
            vector=None
        )
        self.memory_manager.add_toolbox_entry([entry])

    def augment_all(self):
        """Iterates through registered tools and enriches them via LLM."""
        for name, data in self.tools.items():
            print(f"Augmenting {name}...")
            self.tools[name]["augmented_metadata"] = self._get_llm_augmentation(data)

    def save_all_tools_to_db(self):
        """Saves all tools with their augmented metadata to the database."""
        for name in self.tools.keys():
            self._save_tool_to_db(name)

    def _get_llm_augmentation(self, tool_data: Dict) -> Dict:
        prompt = f"""
        Analyze this Python function and its Pydantic parameter schema.
        Generate a detailed 'search_doc' that includes:
        1. A comprehensive description of WHAT it does.
        2. WHEN it should be used (context).
        3. 5 diverse user queries that would trigger this tool.
        
        CODE:
        {tool_data['source_code']}
        
        SCHEMA:
        {json.dumps(tool_data['schema'], indent=2)}
        """
        
        # Using the newer beta.chat.completions.parse for structured metadata
        class ToolMetadata(BaseModel):
            search_doc: str
            trigger_queries: list[str]

        response = self.client.generate_structured(
            messages=[{"role": "system", "content": "You are a technical documentation assistant."},
                      {"role": "user", "content": prompt}],
            response_format=ToolMetadata
            
        )
        print("TempResponse from LLM:", response)
        return response.model_dump()



In [70]:
registry = ToolRegistry(client=openai_client, memory_manager=memory_manager)

class OrderParams(BaseModel):
    symbol: str
    qty: int

@registry.register(OrderParams)
def place_market_order(symbol: str, qty: int):
    """Executes a market order on the exchange."""
    # Your trading logic here
    return f"Order placed for {qty} shares of {symbol}"

# generate some sample tools to populate the registry
class WeatherParams(BaseModel):
    location: str

@registry.register(WeatherParams)
def get_weather(location: str):
    """Fetches current weather for a given location."""
    # Your weather fetching logic here
    return f"Current weather in {location} is Sunny, 25°C"

class NewsParams(BaseModel):
    topic: str


@registry.register(NewsParams)
def get_news(topic: str):
    """Retrieves latest news articles on a specified topic."""
    # Your news fetching logic here
    return f"Latest news on {topic}: 'AI breakthrough in natural language processing!'"


class StockParams(BaseModel):
    ticker: str
@registry.register(StockParams)
def get_stock_price(ticker: str):
    """Gets the current stock price for a given ticker symbol."""
    # Your stock price fetching logic here
    return f"The current stock price of {ticker} is $150.00"    


class RecipeParams(BaseModel):
    dish: str
@registry.register(RecipeParams)

def get_recipe(dish: str):
    """Provides a recipe for a specified dish."""
    # Your recipe fetching logic here
    return f"Recipe for {dish}: 1. Gather ingredients... 2. Cook... 3. Serve!"

# Run the augmentation pipeline
registry.augment_all()
registry.save_all_tools_to_db()

Augmenting place_market_order...
TempResponse from LLM: search_doc='The `place_market_order` function is designed to execute a market order on a trading exchange. It takes two parameters: `symbol`, which is a string representing the stock or asset symbol (e.g., \'AAPL\' for Apple Inc.), and `qty`, which is an integer indicating the quantity of shares to be purchased or sold. The function encapsulates the trading logic necessary to place the order and returns a confirmation message indicating the number of shares ordered and the symbol of the asset.\n\n### When to Use\nThis function should be used in scenarios where a user wants to execute a market order for a specific asset on a trading platform. It is particularly useful for traders who need to quickly buy or sell shares without specifying a limit price, as market orders are executed at the current market price. This function is suitable for automated trading systems, trading bots, or any application that requires real-time trading ca

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
linear.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [71]:
registry.tools.keys()

dict_keys(['place_market_order', 'get_weather', 'get_news', 'get_stock_price', 'get_recipe'])

In [72]:
storage_manager.get_toolbox_table().to_pandas()

,id,thread_id,text,vector,metadata,created_at
0,NaN,NaN,"{""name"": ""place_market_order"", ""schema"": {""typ...","[-0.04271321, 0.27280086, 0.35623002, -0.17683...","[{'key': 'tool_name', 'value': 'place_market_o...",2026-04-12 21:19:37.880585
1,NaN,NaN,"{""name"": ""get_weather"", ""schema"": {""type"": ""fu...","[0.03505471, 0.17138682, 0.64175606, 0.0637802...","[{'key': 'tool_name', 'value': 'get_weather'}]",2026-04-12 21:19:39.576419
2,NaN,NaN,"{""name"": ""get_news"", ""schema"": {""type"": ""funct...","[0.092488475, 0.19871934, 0.39589527, 0.061008...","[{'key': 'tool_name', 'value': 'get_news'}]",2026-04-12 21:19:41.336052
3,NaN,NaN,"{""name"": ""get_stock_price"", ""schema"": {""type"":...","[-0.064055994, 0.24704912, 0.3129562, 0.096422...","[{'key': 'tool_name', 'value': 'get_stock_pric...",2026-04-12 21:19:42.896266
4,NaN,NaN,"{""name"": ""get_recipe"", ""schema"": {""type"": ""fun...","[0.19260955, 0.43130243, 0.29611012, -0.199398...","[{'key': 'tool_name', 'value': 'get_recipe'}]",2026-04-12 21:19:44.573046


In [75]:
tools = memory_manager.read_toolbox_entry(query="market order and weather", limit=3)

In [76]:
for tool in tools:
    # print(registry.tools.get(tool)])
    print(tool)

get_weather
place_market_order
get_news


In [44]:
sample_data = [
    KnowledgeBaseEntry(thread_id="thread1", text="This is a knowledge base entry.", vector=None),

    # add another about USA and its capital
    KnowledgeBaseEntry(thread_id="thread3", text="The capital of USA is Washington D.C.", vector=None),
    KnowledgeBaseEntry(thread_id="thread4", text="The capital of France is Paris.", vector=None),
    # food related
    KnowledgeBaseEntry(thread_id="thread5", text="Pizza is a popular Italian dish.", vector=None),
    KnowledgeBaseEntry(thread_id="thread6", text="Sushi is a traditional Japanese dish.", vector=None),
    # maths related
    KnowledgeBaseEntry(thread_id="thread7", text="The Pythagorean theorem states that a^2 + b^2 = c^2.", vector=None),
    KnowledgeBaseEntry(thread_id="thread8", text="The derivative of sin(x) is cos(x).", vector=None),
    # Climate related
    KnowledgeBaseEntry(thread_id="thread9", text="Climate change is caused by human activities.", vector=None),
    KnowledgeBaseEntry(thread_id="thread10", text="Renewable energy sources include solar and wind power.", vector=None)
]

memory_manager.add_knowledge_base_entry(sample_data)

In [51]:
storage_manager.get_knowledge_base_table().create_fts_index("text")  # Create a fts index before the hybrid search

In [ ]:
storage_manager.get_knowledge_base_table().search(
    query="capital of USA",
    query_type="vector",
    vector_column_name="vector",
).to_pandas().sort_values("_distance").head(5)

,id,thread_id,text,vector,metadata,created_at,_distance
0,None,thread3,The capital of USA is Washington D.C.,"[-0.18850334, -0.052186165, 0.25111595, 0.2473...",None,2026-04-10 12:35:27.608507,15.883088
1,None,thread4,The capital of France is Paris.,"[-0.37511176, -0.11901695, 0.39168936, -0.1172...",None,2026-04-10 12:35:27.608510,58.187855
2,None,thread7,The Pythagorean theorem states that a^2 + b^2 ...,"[-0.025230255, 0.40261218, 0.22764438, 0.14378...",None,2026-04-10 12:35:27.608515,87.320442
3,None,thread8,The derivative of sin(x) is cos(x).,"[0.10524598, 0.29805914, 0.16634221, -0.084207...",None,2026-04-10 12:35:27.608516,89.364426
4,None,thread1,This is a knowledge base entry.,"[0.2393431, 0.024083342, 0.1749246, 0.3335799,...",None,2026-04-10 12:35:27.608498,90.254532
5,None,thread9,Climate change is caused by human activities.,"[0.32795718, 0.16230252, 0.30798298, -0.128249...",None,2026-04-10 12:35:27.608517,91.749405
6,None,thread5,Pizza is a popular Italian dish.,"[-0.3051119, -0.16787319, 0.35846287, -0.70492...",None,2026-04-10 12:35:27.608511,95.047256
7,None,thread10,Renewable energy sources include solar and win...,"[0.3053403, 0.28725216, 0.35977346, -0.0738105...",None,2026-04-10 12:35:27.608519,99.159424
8,None,thread6,Sushi is a traditional Japanese dish.,"[-0.2109578, -0.022132153, -0.021529319, -0.13...",None,2026-04-10 12:35:27.608513,111.797211
